# Import packages and configure reproducible sampling

In [ ]:
import json
import hashlib
import os
import random
import sys
import pandas as pd
import openai
from pathlib import Path

# Load credentials before importing TinyTroupe; its embedding client is created at import time.
env_path = Path("openai.env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            value = value.strip()
            if len(value) >= 2 and value[0] == value[-1] and value[0] in {"'", '"'}:
                value = value[1:-1]
            os.environ[key.strip()] = value

import tinytroupe
from tinytroupe import config_manager
from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld

In [ ]:
RANDOM_SEED = 1
USE_CACHE = True  # Reuse valid population, name, and policy-response caches.
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "Add your key to openai.env as OPENAI_API_KEY=... and rerun this cell."
    )

# Short experiments do not need costly semantic-memory consolidation.
config_manager.update("enable_memory_consolidation", False)
config_manager.update("enable_continuous_contextual_semantic_memory_retrieval", False)
TinyPerson.MAX_EPISODE_LENGTH = 1000

# Prepare a sample from the general Canadian Population

In [ ]:
demographic_source_path = Path("data/canada_population.json")
piaac_source_path = Path("data/canada_piaac.json")
cache_path = Path("cache/canada_census_population.json")
cache_path.parent.mkdir(parents=True, exist_ok=True)

population_size = 100
sample_size = 25

if sample_size > population_size:
    raise ValueError("sample_size cannot exceed population_size")

demographic_source_text = demographic_source_path.read_text(encoding="utf-8")
piaac_source_text = piaac_source_path.read_text(encoding="utf-8")
source_hash = hashlib.sha256(
    (demographic_source_text + piaac_source_text).encode()
).hexdigest()
demographic_source = json.loads(demographic_source_text)
piaac_source = json.loads(piaac_source_text)
demographic_dimensions = demographic_source["dimensions"]
piaac_dimensions = piaac_source["dimensions"]

def weighted_choice(rng, distribution):
    labels = list(distribution)
    weights = [distribution[label] for label in labels]
    return rng.choices(labels, weights=weights, k=1)[0]

demographic_distributions = {
    name: specification["categories"]
    for name, specification in demographic_dimensions.items()
}
piaac_distributions = {
    name: specification["categories"]
    for name, specification in piaac_dimensions.items()
}

proficiency_scores = {
    "Below Level 1": 0.05, "Level 1": 0.20, "Level 2": 0.40,
    "Level 3": 0.65, "Level 4": 0.85, "Level 5": 1.00,
}
category_scores = {
    "Category 1 - lowest": 0.10, "Category 2": 0.30, "Category 3": 0.50,
    "Category 4": 0.70, "Category 5 - highest": 0.90,
}
curiosity_scores = {
    "Lower - below -0.5 SD": 0.20, "Middle - -0.5 to 0.5 SD": 0.50,
    "Higher - above 0.5 SD": 0.80,
}
education_scores = {
    "Below high school completion": 0.20,
    "High school or other non-university credential": 0.55,
    "College or university credential": 0.85,
}

def ai_literacy_traits(profile, rng):
    literacy = proficiency_scores[profile["literacy_proficiency"]]
    numeracy = proficiency_scores[profile["numeracy_proficiency"]]
    problem_solving = proficiency_scores[profile["adaptive_problem_solving_proficiency"]]
    technology = (
        category_scores[profile["ict_use_at_home"]]
        + category_scores[profile["ict_use_at_work"]]
    ) / 2
    curiosity = curiosity_scores[profile["curiosity"]]
    learning = (
        0.45 * curiosity
        + 0.35 * category_scores[profile["learning_at_work"]]
        + 0.20 * (profile["recent_nonformal_education"] == "Participated")
    )
    education = education_scores[profile["education_attainment"]]

    def estimate(weighted_foundation, residual_sd=0.14):
        score = min(1, max(0, weighted_foundation + rng.gauss(0, residual_sd)))
        label = "Low" if score < 0.35 else "Moderate" if score < 0.65 else "High"
        return f"{label} ({round(score * 100)}/100; synthetic estimate)"

    return {
        "functional_ai_knowledge": estimate(
            0.32 * literacy + 0.18 * problem_solving + 0.20 * technology
            + 0.15 * education + 0.15 * learning
        ),
        "applied_ai_use_competence": estimate(
            0.10 * literacy + 0.15 * problem_solving + 0.40 * technology
            + 0.20 * learning + 0.15 * education
        ),
        "critical_ai_evaluation_capacity": estimate(
            0.40 * literacy + 0.25 * problem_solving + 0.15 * numeracy
            + 0.10 * technology + 0.10 * curiosity
        ),
        # PIAAC has no direct ethics measure, so this estimate has greater residual variance.
        "ethical_ai_understanding": estimate(
            0.25 * literacy + 0.20 * education + 0.15 * curiosity
            + 0.10 * learning + 0.30 * rng.random(),
            residual_sd=0.18,
        ),
    }

algorithm_version = 4
if USE_CACHE and cache_path.exists():
    cache = json.loads(cache_path.read_text(encoding="utf-8"))
else:
    cache = {}

cache_is_valid = (
    cache.get("source_hash") == source_hash
    and cache.get("algorithm_version") == algorithm_version
    and len(cache.get("profiles", [])) >= population_size
)

if cache_is_valid:
    demographic_population = cache["profiles"]
    print(f"Loaded {len(demographic_population)} cached demographic and PIAAC profiles.")
else:
    demographic_rng = random.Random(RANDOM_SEED)
    piaac_rng = random.Random(RANDOM_SEED + 10)
    ai_trait_rng = random.Random(RANDOM_SEED + 20)
    demographic_population = []

    for index in range(population_size):
        profile = {
            "profile_id": f"CA-SYN-{index + 1:06d}",
            "country": "Canada",
            **{
                name: weighted_choice(demographic_rng, distribution)
                for name, distribution in demographic_distributions.items()
            },
            **{
                name: weighted_choice(piaac_rng, distribution)
                for name, distribution in piaac_distributions.items()
            },
        }
        profile.update(ai_literacy_traits(profile, ai_trait_rng))
        demographic_population.append(profile)

    cache = {
        "source_hash": source_hash,
        "algorithm_version": algorithm_version,
        "random_seed": RANDOM_SEED,
        "profiles": demographic_population,
    }
    cache_path.write_text(
        json.dumps(cache, indent=2),
        encoding="utf-8",
    )

sampling_pool = demographic_population[:population_size]
sample_rng = random.Random(RANDOM_SEED + 1)
demographic_sample = sample_rng.sample(sampling_pool, k=sample_size)
pd.DataFrame(demographic_sample)

In [ ]:
names_cache_path = Path("cache/canada_profile_names.json")
if USE_CACHE and names_cache_path.exists():
    names_by_profile = json.loads(names_cache_path.read_text(encoding="utf-8"))
else:
    names_by_profile = {}

naming_profiles = {
    profile["profile_id"]: {
        key: value
        for key, value in profile.items()
        if key in {"profile_id", "country", *demographic_dimensions}
    }
    for profile in demographic_sample
}
profile_keys = {
    profile_id: hashlib.sha256(
        json.dumps(profile, sort_keys=True).encode()
    ).hexdigest()
    for profile_id, profile in naming_profiles.items()
}
missing_profiles = [
    naming_profiles[profile["profile_id"]]
    for profile in demographic_sample
    if profile_keys[profile["profile_id"]] not in names_by_profile
]

if missing_profiles:
    prompt = f"""
Assign one unique, plausible full name to each synthetic Canadian profile below.
Return only a JSON object with a 'names' object mapping each profile_id to its name.
Do not add titles, explanations, or extra IDs. Avoid caricatures and stereotypes.
Profiles: {json.dumps(missing_profiles, indent=2)}
"""
    client = openai.OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=config_manager.get("model"),
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    generated_names = json.loads(response.choices[0].message.content)["names"]

    expected_ids = {profile["profile_id"] for profile in missing_profiles}
    if set(generated_names) != expected_ids:
        raise RuntimeError("The generated-name response did not match the requested profiles.")
    if len(set(generated_names.values())) != len(generated_names):
        raise RuntimeError("The generated names were not unique.")

    for profile_id, name in generated_names.items():
        names_by_profile[profile_keys[profile_id]] = name
    names_cache_path.write_text(
        json.dumps(names_by_profile, indent=2),
        encoding="utf-8",
    )

TinyPerson.clear_agents()
tiny_people = []
for profile in demographic_sample:
    profile_key = profile_keys[profile["profile_id"]]
    person_name = names_by_profile[profile_key]
    agent = TinyPerson.get_agent_by_name(person_name)
    if agent is None:
        agent = TinyPerson(person_name)

    agent.define("nationality", "Canadian")
    agent.define("country_of_residence", "Canada")

    for field, value in profile.items():
        if field not in {"profile_id", "country"}:
            agent.define(field, value)

    tiny_people.append(agent)

[(person.name, person._persona) for person in tiny_people]

# Policy announcement simulation

Start with one policy to verify the workflow and API cost. Replace `selected_policy_ids` with `list(policy_data["announcements"])` to run all policies. Each selected policy requires one response from every agent.

In [ ]:
policy_data = json.loads(
    Path("data/policy_announcements.json").read_text(encoding="utf-8")
)
strategic_landscape = json.loads(
    Path("data/canada_ai_strategic_landscape.json").read_text(encoding="utf-8")
)
landscape_for_simulation = {
    **strategic_landscape,
    "challenges": [
        {"name": challenge["name"], "context": challenge["context"]}
        for challenge in strategic_landscape["challenges"]
    ],
}

selected_policy_ids = ["ai_literacy"]
# selected_policy_ids = list(policy_data["announcements"])

unknown_policy_ids = set(selected_policy_ids) - set(policy_data["announcements"])
if unknown_policy_ids:
    raise KeyError(f"Unknown policy IDs: {sorted(unknown_policy_ids)}")

pd.DataFrame(
    [
        {"policy_id": policy_id, **policy_data["announcements"][policy_id]}
        for policy_id in selected_policy_ids
    ]
)

In [ ]:
class OneTalkWorld(TinyWorld):
    """Deliver at most one correctly targeted TALK action per agent and round."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.expected_targets = {}

    def _handle_actions(self, source, actions):
        expected = self.expected_targets.get(source.name)
        filtered_actions = []
        talk_kept = False
        for action in actions:
            if action.get("type") != "TALK":
                filtered_actions.append(action)
            elif not talk_kept and (action.get("target") or "") == expected:
                filtered_actions.append(action)
                talk_kept = True
        return super()._handle_actions(source, filtered_actions)

TinyWorld.clear_environments()
world = OneTalkWorld(
    "Canadian AI policy announcements",
    tiny_people,
    broadcast_if_no_target=False,
)
minimum_known_challenges = 2
maximum_known_challenges = 5
awareness_rng = random.Random(RANDOM_SEED + 30)
agent_strategic_awareness = {}

# Information is unevenly distributed: each agent knows only a reproducible random subset.
for agent in tiny_people:
    number_known = awareness_rng.randint(
        minimum_known_challenges, maximum_known_challenges
    )
    known_challenges = awareness_rng.sample(
        strategic_landscape["challenges"], k=number_known
    )
    agent_strategic_awareness[agent.name] = [
        challenge["name"] for challenge in known_challenges
    ]
    personal_context = (
        f"Background information you happen to know about {strategic_landscape['title']}:\n"
        + "\n".join(
            f"- {challenge['name']}: {challenge['context']}"
            for challenge in known_challenges
        )
        + f"\n\nInterpretation boundary: {strategic_landscape['interpretation_rule']}"
        + "\nUse this only as background when interpreting later announcements. "
          "Do not assume you know the omitted challenges, and do not recite this list. "
          "Let relevant background influence what you notice, prioritize, question, or worry about. "
          "Do not force it into every response or mention it merely to demonstrate knowledge."
    )
    agent.listen(personal_context, source=world, communication_display=False)
discussion_rounds = 3
response_word_limit = 40
network_rng = random.Random(RANDOM_SEED + 2)
social_edges = set()

# Ring-lattice: every agent has four stable local contacts.
for index, agent in enumerate(tiny_people):
    for offset in (1, 2):
        peer = tiny_people[(index + offset) % len(tiny_people)]
        edge = tuple(sorted((agent.name, peer.name)))
        social_edges.add(edge)
        agent.make_agent_accessible(peer, "Social-network contact")
        peer.make_agent_accessible(agent, "Social-network contact")

# Random long-distance ties create a small-world-style network.
for _ in range(max(1, len(tiny_people) // 4)):
    agent, peer = network_rng.sample(tiny_people, 2)
    edge = tuple(sorted((agent.name, peer.name)))
    social_edges.add(edge)
    agent.make_agent_accessible(peer, "Extended social-network contact")
    peer.make_agent_accessible(agent, "Extended social-network contact")

social_edges_df = pd.DataFrame(sorted(social_edges), columns=["agent", "peer"])
neighbours = {person.name: [] for person in tiny_people}
for left, right in sorted(social_edges):
    neighbours[left].append(right)
    neighbours[right].append(left)
for name in neighbours:
    neighbours[name].sort()

world.broadcast_internal_goal(
    "Participate naturally in the policy discussion. Produce exactly one TALK action per round. "
    "Always speak in English. Give an independent reaction first; address a named participant only in later rounds. "
    f"Keep spoken responses conversational and within {response_word_limit} words."
)
policy_responses = []

response_cache_path = Path("cache/canada_strategic_context_responses.json")
signature_payload = {
    "policy_ids": selected_policy_ids,
    "policies": [policy_data["announcements"][key] for key in selected_policy_ids],
    "strategic_landscape": landscape_for_simulation,
    "agent_strategic_awareness": agent_strategic_awareness,
    "known_challenge_range": [minimum_known_challenges, maximum_known_challenges],
    "agents": [person._persona for person in tiny_people],
    "social_edges": sorted(social_edges),
    "discussion_rounds": discussion_rounds,
    "response_word_limit": response_word_limit,
    "prompt_version": 8,
}
simulation_signature = hashlib.sha256(
    json.dumps(signature_payload, sort_keys=True).encode()
).hexdigest()

if USE_CACHE and response_cache_path.exists():
    response_cache = json.loads(response_cache_path.read_text(encoding="utf-8"))
else:
    response_cache = {}

if response_cache.get("signature") == simulation_signature:
    policy_responses = response_cache["policy_responses"]
    print("Loaded cached policy responses.")
else:
    for policy_id in selected_policy_ids:
        announcement = policy_data["announcements"][policy_id]
        stimulus = f"""
Government announcement: {announcement['title']}

{announcement['text']}

Discuss this announcement as a Canadian citizen. Focus on what genuinely matters to you;
you do not need to cover a fixed checklist of reaction, impact, support, and concerns.
"""
        world.broadcast(stimulus)
        actions_by_round = []
        reply_context_by_round = []
        previous_statements = {}
        for round_index in range(discussion_rounds):
            expected_targets = {}
            round_reply_context = {}
            for agent_index, agent in enumerate(tiny_people):
                if round_index == 0:
                    target = ""
                    round_instruction = (
                        "Before responding, privately consider every strategic challenge you were given. "
                        "If one materially affects your reaction, weave that connection concretely and naturally "
                        "into what you notice, prioritize, question, or worry about. If none is relevant, do not force one. "
                        "Give your own immediate, gut response to the policy. Do not name, address, "
                        "agree with, or reply to another participant. Use an empty TALK target. "
                    )
                else:
                    contacts = neighbours[agent.name]
                    target = contacts[(agent_index + round_index - 1) % len(contacts)]
                    target_statement = previous_statements[target]
                    round_reply_context[agent.name] = {
                        "target": target,
                        "statement": target_statement,
                    }
                    round_instruction = (
                        f"Respond directly to this statement made by {target} in the preceding round:\n"
                        f"\"{target_statement}\"\n"
                        "Privately consider whether any strategic challenge you know materially sharpens your reply. "
                        "When relevant, incorporate its substance naturally; otherwise do not force it. "
                        "Engage with one concrete idea from that statement by building on it, questioning it, "
                        "contrasting it, or disagreeing. Do not merely repeat your own earlier view. "
                    )
                expected_targets[agent.name] = target
                agent.internalize_goal(
                    round_instruction
                    + f"Your single TALK action must target exactly '{target}' and stay within {response_word_limit} words."
                )
            world.expected_targets = expected_targets
            raw_round_actions = world.run(
                    1,
                    return_actions=True,
                    randomize_agents_order=False,
                    parallelize=True,
                )[0]
            round_actions = {}
            current_statements = {}
            for agent_name, actions in raw_round_actions.items():
                matching_talk_actions = [
                    action for action in actions
                    if action.get("action", action).get("type") == "TALK"
                    and (action.get("action", action).get("target") or "") == expected_targets[agent_name]
                ]
                if not matching_talk_actions:
                    raise RuntimeError(
                        f"{agent_name} produced no TALK action targeting "
                        f"{expected_targets[agent_name]!r} in round {round_index + 1}."
                    )
                selected_action = matching_talk_actions[0]
                round_actions[agent_name] = [selected_action]
                selected_talk = selected_action.get("action", selected_action)
                current_statements[agent_name] = str(selected_talk.get("content", ""))
            actions_by_round.append(round_actions)
            reply_context_by_round.append(round_reply_context)
            previous_statements = current_statements
        policy_responses.append(
            {
                "policy_id": policy_id,
                "policy_title": announcement["title"],
                "policy_text": announcement["text"],
                "actions_by_round": actions_by_round,
                "reply_context_by_round": reply_context_by_round,
            }
        )

    response_cache_path.write_text(
        json.dumps(
            {
                "signature": simulation_signature,
                "policy_responses": policy_responses,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

In [ ]:
challenge_by_name = {
    challenge["name"]: challenge
    for challenge in strategic_landscape["challenges"]
}

def detect_invoked_challenges(agent_name, response):
    response_lower = response.lower()
    return [
        challenge_name
        for challenge_name in agent_strategic_awareness.get(agent_name, [])
        if any(
            signal.lower() in response_lower
            for signal in challenge_by_name[challenge_name].get("signals", [])
        )
    ]

response_rows = []
for result in policy_responses:
    for round_number, round_actions in enumerate(result["actions_by_round"], start=1):
        reply_context = result.get("reply_context_by_round", [{}] * len(result["actions_by_round"]))[round_number - 1]
        for agent_name, actions in round_actions.items():
            for action in actions:
                action_data = action.get("action", action)
                cognitive_state = action.get("cognitive_state", {})
                response_text = " ".join(
                    str(action_data.get("content", "")).split()[:response_word_limit]
                )
                response_rows.append(
                    {
                        "policy_id": result["policy_id"],
                        "policy_title": result["policy_title"],
                        "policy_text": result.get("policy_text", ""),
                        "round": round_number,
                        "agent_name": agent_name,
                        "action_type": action_data.get("type"),
                        "target": action_data.get("target"),
                        "reply_to_statement": reply_context.get(agent_name, {}).get("statement", ""),
                        "known_challenges": agent_strategic_awareness.get(agent_name, []),
                        "invoked_challenges": detect_invoked_challenges(agent_name, response_text),
                        "response": response_text,
                        "original_word_count": len(
                            str(action_data.get("content", "")).split()
                        ),
                        "within_word_limit": len(
                            str(action_data.get("content", "")).split()
                        ) <= response_word_limit,
                        "attention": cognitive_state.get("attention"),
                        "emotions": cognitive_state.get("emotions"),
                    }
                )

responses_df = pd.DataFrame(response_rows)
responses_df

In [ ]:
outputs_path = Path("outputs/strategic_context")
outputs_path.mkdir(parents=True, exist_ok=True)
responses_df.to_excel(outputs_path / "ai_literacy.xlsx", index=False)

# Animated social-network conversation

The notebook displays a rotating group conversation. Hover over any demographic-aware figure for their profile; the active speech appears above the group.

In [ ]:
import html
from IPython.display import HTML, display

talk_events = responses_df.loc[
    responses_df["action_type"].eq("TALK"),
    ["policy_id", "policy_title", "policy_text", "round", "agent_name", "target", "reply_to_statement", "known_challenges", "invoked_challenges", "response", "attention"],
] .fillna("").to_dict(orient="records")

payload = {
    "events": talk_events,
    "profiles": {
    person.name: profile for person, profile in zip(tiny_people, demographic_sample)
    },
    "names": [person.name for person in tiny_people],
    "strategic_awareness": agent_strategic_awareness,
    "network": {
        "participants": len(tiny_people),
        "relationships": len(social_edges),
        "average_contacts": round(sum(map(len, neighbours.values())) / len(neighbours), 1),
        "minimum_contacts": min(map(len, neighbours.values())),
        "maximum_contacts": max(map(len, neighbours.values())),
        "density": round(2 * len(social_edges) / (len(tiny_people) * (len(tiny_people) - 1)), 3),
        "structure": "Four local contacts per participant plus random long-distance ties",
    },
}
payload_json = json.dumps(payload).replace("</", "<\\/")
document = r'''<!doctype html><meta charset="utf-8">
<style>
:root{--talk:#00cd00;--think:#008000;--conversation:#00ffff;--reach:#800080;--done:#d1d1d1}*{box-sizing:border-box}body{margin:0;background:linear-gradient(145deg,#fffaf1,#f5fbff);color:#352f2a;font:14px system-ui,sans-serif}.stage{max-width:1100px;margin:auto;padding:20px;text-align:center}.eyebrow{color:#766b62;font-size:11px;font-weight:700;letter-spacing:.13em;text-transform:uppercase}.network{display:flex;gap:8px;justify-content:center;flex-wrap:wrap;margin:10px auto}.stat{min-width:110px;padding:7px 11px;border:1px solid #ddd4ca;border-radius:10px;background:#ffffffc9}.stat b{display:block;font-size:17px;color:#5f276d}.network-note{color:#766b62;font-size:11px;margin-top:-3px}.policy{max-width:820px;margin:10px auto;padding:12px 16px;border-left:5px solid var(--reach);border-radius:8px;background:#fff;text-align:left;line-height:1.4;box-shadow:0 3px 12px #0001}.policy summary{cursor:pointer;font-weight:750;color:#5f276d}.policy div{margin-top:8px;white-space:pre-wrap}.reply{max-width:680px;margin:10px auto -10px;padding:10px 15px;border-radius:14px;background:#e8ffff;border:2px solid var(--conversation);text-align:left;font-size:12px}.legend{display:flex;gap:14px;justify-content:center;flex-wrap:wrap;margin:9px 0;font-size:12px}.dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:4px}.progress{height:5px;max-width:760px;margin:10px auto;background:#e5e7eb;border-radius:9px;overflow:hidden}.progress i{display:block;height:100%;width:0;background:linear-gradient(90deg,var(--talk),var(--conversation));transition:width .35s}.controls{display:flex;gap:8px;align-items:center;justify-content:center}.controls input{width:min(600px,60vw)}button{padding:7px 13px;border:1px solid #c9c2b9;border-radius:8px;background:white;cursor:pointer}.title{font-size:22px;font-weight:750}.meta{color:#766b62;margin:4px}.speech{min-height:78px;max-width:720px;margin:20px auto 12px;padding:17px 20px;border-radius:22px;background:#f3fff3;border:3px solid var(--talk);text-align:left;font-size:15px;line-height:1.45;box-shadow:0 6px 20px #0002}.speech.pulse{animation:pop .35s ease-out}.people{display:flex;justify-content:space-around;align-items:end;min-height:245px}.person{position:relative;width:145px;padding:9px;border:3px solid transparent;border-radius:18px;transition:transform .25s,opacity .25s}.person:not(.speaker):not(.target){opacity:.68}.person.speaker{background:#eaffea;border-color:var(--talk);transform:translateY(-9px)}.person.target{background:#e8ffff;border-color:var(--conversation)}.icon{display:block;font-size:58px}.speaker .icon{animation:bob 1s ease-in-out infinite alternate}.role{min-height:16px;font-size:9px;font-weight:800;letter-spacing:.08em}.speaker .role{color:#007a00}.target .role{color:#007f87}.name{font-weight:700;font-size:12px}.tip{display:none;position:absolute;z-index:10;bottom:100%;left:50%;transform:translateX(-50%);width:245px;padding:10px;background:#26211e;color:white;border-radius:9px;text-align:left;font-size:12px;box-shadow:0 5px 18px #0005}.person:hover{opacity:1}.person:hover .tip{display:block}.hint{margin:10px;color:#857970;font-size:11px}@keyframes bob{to{transform:translateY(-5px) rotate(2deg)}}@keyframes pop{0%{transform:scale(.97);opacity:.4}100%{transform:scale(1);opacity:1}}@media(max-width:720px){.people{overflow-x:auto;justify-content:flex-start;gap:8px}.person{min-width:120px}.speech{font-size:13px}}
.invoked{max-width:720px;margin:12px auto -10px;color:#5f276d;font-size:12px}.invoked b{margin-right:6px}.challenge-chip{display:inline-block;margin:3px;padding:4px 8px;border-radius:12px;background:#f4e8f7;border:1px solid var(--reach)}
</style><div class="stage"><div class="eyebrow">TinyTroupe policy simulation</div><div class="title" id="title"></div><div class="network" id="network"></div><div class="network-note" id="network-note"></div><details class="policy" open><summary>Policy announcement</summary><div id="policy"></div></details><div class="meta" id="meta"></div><div class="progress"><i id="bar"></i></div><div class="legend"><span><i class="dot" style="background:var(--talk)"></i>TALK</span><span><i class="dot" style="background:var(--think)"></i>THINK</span><span><i class="dot" style="background:var(--conversation)"></i>CONVERSATION</span><span><i class="dot" style="background:var(--reach)"></i>REACH_OUT</span><span><i class="dot" style="background:var(--done)"></i>DONE</span></div><div class="reply" id="reply"></div><div class="speech" id="speech"></div><div class="people" id="people"></div><div class="controls"><button id="prev" title="Previous message">◀</button><button id="play">▶ Play</button><button id="next" title="Next message">▶</button><input id="timeline" type="range" min="0" value="0" aria-label="Conversation timeline"><span id="count"></span></div><div class="hint">Hover over a person for demographics · use ← and → to navigate · green speaks, cyan listens</div></div>
<script>
const data=__PAYLOAD__,events=data.events,recent=[];let index=0,timer=null;
const esc=s=>String(s??'').replace(/[&<>\"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','\"':'&quot;'}[c]));
const invokedBox=document.createElement('div');invokedBox.id='invoked';invokedBox.className='invoked';document.getElementById('speech').before(invokedBox);
const net=data.network;document.getElementById('network').innerHTML=[['Participants',net.participants],['Relationships',net.relationships],['Average contacts',net.average_contacts],['Contact range',net.minimum_contacts+'–'+net.maximum_contacts],['Density',net.density]].map(([label,value])=>'<div class="stat"><b>'+esc(value)+'</b>'+esc(label)+'</div>').join('');document.getElementById('network-note').textContent=net.structure;
function profile(name){const p=data.profiles[name]||{},known=data.strategic_awareness[name]||[];return '<b>'+esc(name)+'</b><br>'+Object.entries(p).filter(([k])=>k!=='profile_id').map(([k,v])=>esc(k.replaceAll('_',' ').replace(/\\b\\w/g,c=>c.toUpperCase()))+': '+esc(v)).join('<br>')+'<hr><b>Known strategic challenges</b><br>'+known.map(esc).join('<br>')}
function avatar(name){const p=data.profiles[name]||{},race=p.race_ethnicity||'',gender=String(p.gender||'').toLowerCase();const tone=race==='Black'?'🏿':['South Asian','Indigenous','Filipino','Arab','Latin American'].includes(race)?'🏽':['White','Chinese'].includes(race)?'🏻':'';const base=gender.includes('woman')||gender.includes('female')?'👩':gender.includes('man')||gender.includes('male')?'👨':'🧑';return base+tone}
function render(){const e=events[index],candidates=[e.agent_name,e.target,...recent.slice().reverse(),...data.names],visible=[];for(const n of candidates){if(n&&!visible.includes(n))visible.push(n);if(visible.length===Math.min(6,data.names.length))break}recent.push(e.agent_name);if(recent.length>8)recent.shift();document.getElementById('title').textContent=e.policy_title;document.getElementById('policy').textContent=e.policy_text;document.getElementById('meta').textContent='Round '+e.round+' · '+(e.target?e.agent_name+' → '+e.target:'independent reaction by '+e.agent_name)+' · message '+(index+1)+'/'+events.length;const reply=document.getElementById('reply');reply.style.display=e.reply_to_statement?'block':'none';reply.innerHTML=e.reply_to_statement?'↩ <b>Responding to '+esc(e.target)+':</b> “'+esc(e.reply_to_statement)+'”':'';const speech=document.getElementById('speech');speech.innerHTML='💬 <b>'+esc(e.agent_name)+'</b><br>'+esc(e.response);speech.classList.remove('pulse');void speech.offsetWidth;speech.classList.add('pulse');document.getElementById('people').innerHTML=visible.map(n=>'<div class="person '+(n===e.agent_name?'speaker':n===e.target?'target':'')+'"><span class="icon">'+avatar(n)+'</span><div class="role">'+(n===e.agent_name?'SPEAKING':n===e.target?'LISTENING':'IN THE GROUP')+'</div><div class="name">'+esc(n)+'</div><div class="tip">'+profile(n)+'</div></div>').join('');document.getElementById('bar').style.width=((index+1)/events.length*100)+'%';timeline.value=index;count.textContent=(index+1)+'/'+events.length}
const baseRender=render;render=function(){baseRender();const invoked=events[index].invoked_challenges||[];invokedBox.innerHTML=invoked.length?'<b>Possible strategic context invoked:</b>'+invoked.map(x=>'<span class="challenge-chip">'+esc(x)+'</span>').join(''):'<span>No clear strategic-context signal in this response</span>'}
const timeline=document.getElementById('timeline'),count=document.getElementById('count');timeline.max=Math.max(0,events.length-1);timeline.oninput=()=>{index=+timeline.value;render()};prev.onclick=()=>{index=(index-1+events.length)%events.length;render()};next.onclick=()=>{index=(index+1)%events.length;render()};play.onclick=()=>{if(timer){clearInterval(timer);timer=null;play.textContent='▶ Play'}else{timer=setInterval(()=>{index=(index+1)%events.length;render()},2200);play.textContent='❚❚ Pause'}};document.onkeydown=e=>{if(e.key==='ArrowLeft')prev.click();if(e.key==='ArrowRight')next.click();if(e.key===' ')play.click()};if(events.length)render();else document.body.textContent='No TALK events are available to animate.';
</script>'''.replace("__PAYLOAD__", payload_json)
standalone_path = outputs_path / "conversation_animation.html"
standalone_path.write_text(document, encoding="utf-8")
display(HTML(
    f'<p><a href="{standalone_path.as_posix()}" target="_blank">Open standalone conversation animation</a></p>'
    f'<p><code>{standalone_path.resolve()}</code></p>'
))
